# Epicenter Analysis — CABIC Cohort

Identical pipeline to `Epicenter.ipynb` (GPR normative modeling → Z-scores → K-means subtyping → GOF epicenter seeding), applied to the **CABIC** cohort. Only the data paths differ.

## Part 1 — GPR normative modeling and Z-score computation

In [ ]:
import os
import pandas as pd
import numpy as np
from multiprocessing import Pool
from tqdm import tqdm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from neuroCombat import neuroCombat
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Configuration (CABIC-specific paths)
# ============================================================
NUM_CPU_CORES = min(30, os.cpu_count() or 8)
MORPHO_FEATURES_PATH = 'data/CABIC_all_aparc_features.csv'
SUBJECT_INFO_PATH = 'data/CABIC_AGE.xlsx'
ZSCORE_OUTPUT_PATH = 'output/CABIC_Morpho_Zscore_aparc.csv'
MORPHO_FEATURES = ['GrayVol']
EULER_COL = 'mean_euler_bh'

def train_gpr_model(args):
    """Train a single GPR model (for multiprocessing)."""
    i, X_td_scaled, Y_td_i, gpr_base, feature_name = args
    try:
        gpr_model = clone(gpr_base)
        gpr_model.fit(X_td_scaled, Y_td_i)
        return i, gpr_model
    except Exception:
        return i, None

def build_normative_model_and_calculate_deviation():
    """ComBat -> QC regression -> GPR on TDC -> Z-scores (CABIC cohort)."""
    print("\n--- Loading data ---")
    df_morpho = pd.read_csv(MORPHO_FEATURES_PATH)
    df_info = pd.read_excel(SUBJECT_INFO_PATH)

    df_morpho['participant_id'] = df_morpho['participant_id'].astype(str).str.strip()
    df_info['SUBID'] = df_info['SUBID'].astype(str).str.strip()

    needed_cols = ['SUBID', 'GROUP', 'AGE', 'SEX', 'SITE', EULER_COL]
    df_all = pd.merge(df_morpho, df_info[needed_cols],
                     left_on='participant_id', right_on='SUBID', how='inner')
    df_all['SEX_code'] = df_all['SEX'].apply(lambda x: 0 if str(x).upper() in ['M', 'MALE'] else 1)
    df_all = df_all.dropna(subset=['AGE', 'SEX_code', 'SITE', EULER_COL])

    all_regions = sorted(df_all['Label'].unique())
    n_regions = len(all_regions)
    all_feature_names = [f"Zscore_{region}_{feat}" for region in all_regions for feat in MORPHO_FEATURES]

    # Reshape to per-subject feature vectors
    print("--- Step 1: reshape feature matrix ---")
    all_subjects_features, combat_ids = [], []
    subject_counts = df_all['SUBID'].value_counts()
    valid_ids = subject_counts[subject_counts == n_regions].index.tolist()
    df_valid = df_all[df_all['SUBID'].isin(valid_ids)]
    for sub_id in tqdm(valid_ids, desc="Reshape"):
        df_sub = df_valid[df_valid['SUBID'] == sub_id].sort_values(by='Label')
        all_subjects_features.append(df_sub.set_index('Label')[MORPHO_FEATURES].values.flatten())
        combat_ids.append(sub_id)

    X_input = np.array(all_subjects_features).T
    df_covars_full = df_valid.drop_duplicates(subset=['SUBID']).set_index('SUBID').loc[combat_ids]
    covars_combat = pd.DataFrame({
        'batch': df_covars_full['SITE'].values,
        'age': df_covars_full['AGE'].values,
        'sex': df_covars_full['SEX_code'].values,
        'group': df_covars_full['GROUP'].values,
    })

    # ComBat harmonization
    print("--- Step 2: ComBat site harmonization ---")
    X_harmonized = neuroCombat(dat=X_input, covars=covars_combat, batch_col='batch',
                               categorical_cols=['sex', 'group'], continuous_cols=['age'])['data'].T

    # Regression against QC metric
    print(f"--- Step 3: regress out {EULER_COL} ---")
    euler_values = df_covars_full[EULER_COL].values.reshape(-1, 1)
    X_residualized = np.zeros_like(X_harmonized)
    for i in range(X_harmonized.shape[1]):
        y = X_harmonized[:, i].reshape(-1, 1)
        reg = LinearRegression().fit(euler_values, y)
        X_residualized[:, i] = (y - reg.predict(euler_values)).flatten() + np.mean(y)

    # GPR normative modeling on TDC
    print("--- Step 4: train GPR models on TDC ---")
    df_final = pd.DataFrame(X_residualized, columns=[f'C{i}' for i in range(X_residualized.shape[1])])
    df_final['SUBID'] = combat_ids
    df_final = pd.merge(df_covars_full.reset_index()[['SUBID', 'GROUP', 'AGE', 'SEX_code']],
                       df_final, on='SUBID')

    df_tdc = df_final[df_final['GROUP'] == 'TDC']
    X_td = df_tdc[['AGE', 'SEX_code']].values
    scaler_X = StandardScaler().fit(X_td)
    X_td_scaled = scaler_X.transform(X_td)
    Y_td = df_tdc[[f'C{i}' for i in range(len(all_feature_names))]].values

    kernel = C(1.0) * RBF(length_scale=[1.0, 1.0]) + WhiteKernel(noise_level=1.0)
    gpr_base = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5,
                                        random_state=42, normalize_y=True)
    pool_inputs = [(i, X_td_scaled, Y_td[:, i], gpr_base, all_feature_names[i])
                  for i in range(len(all_feature_names))]
    with Pool(NUM_CPU_CORES) as pool:
        gpr_models = [m for _, m in tqdm(pool.imap(train_gpr_model, pool_inputs),
                                        total=len(all_feature_names))]

    # Z-score computation
    print("--- Step 5: compute Z-scores ---")
    X_all = df_final[['AGE', 'SEX_code']].values
    X_all_scaled = scaler_X.transform(X_all)
    Y_obs = df_final[[f'C{i}' for i in range(len(all_feature_names))]].values

    Z = np.zeros_like(Y_obs)
    for i in range(len(all_feature_names)):
        mu, sigma = gpr_models[i].predict(X_all_scaled, return_std=True)
        Z[:, i] = (Y_obs[:, i] - mu) / (sigma + 1e-9)

    zscore_df = pd.DataFrame(Z, columns=all_feature_names)
    zscore_df.insert(0, 'SUBID', df_final['SUBID'].values)
    zscore_df.insert(1, 'GROUP', df_final['GROUP'].values)
    os.makedirs(os.path.dirname(ZSCORE_OUTPUT_PATH), exist_ok=True)
    zscore_df.to_csv(ZSCORE_OUTPUT_PATH, index=False)
    print(f"Saved Z-scores to {ZSCORE_OUTPUT_PATH}")
    return zscore_df

zscore_df = build_normative_model_and_calculate_deviation()

In [ ]:
# ============================================================
# Compute the TDC-average MIND fingerprint (68x68)
# Read every TDC subject's MIND_network_aparc.csv via the
# 'aparc' path column in data/CABIC_AGE.xlsx and average the
# matrices into a single group fingerprint used for the GOF
# epicenter seeding below.
# ============================================================
import os
import numpy as np
import pandas as pd

SUBJECT_INFO_PATH = 'data/CABIC_AGE.xlsx'
TDC_AVG_OUTPUT_PATH = 'output/TDC_Average_MIND_CABIC_aparc.csv'
os.makedirs(os.path.dirname(TDC_AVG_OUTPUT_PATH), exist_ok=True)

df_info = pd.read_excel(SUBJECT_INFO_PATH)
df_tdc = df_info[df_info['GROUP'] == 'TDC'].dropna(subset=['aparc'])

matrices, region_names = [], None
for _, row in df_tdc.iterrows():
    mat = pd.read_csv(row['aparc']).values                # 68x68 MIND matrix
    if region_names is None:
        region_names = pd.read_csv(row['aparc']).columns.tolist()
    matrices.append(mat)

avg_mind = np.mean(np.array(matrices), axis=0)            # group average (68x68)
pd.DataFrame(avg_mind, columns=region_names).to_csv(TDC_AVG_OUTPUT_PATH, index=False)
print(f"Averaged {len(matrices)} TDC MIND matrices -> {TDC_AVG_OUTPUT_PATH}")

## Part 2 — K-means subtyping

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import os

ZSCORE_FILE_PATH = 'output/CABIC_Morpho_Zscore_aparc.csv'
K_CLUSTERS = 2
OUTPUT_DIR = 'output/SubLabel'
FEATURE_SUBSET = ['Vol']
FEATURE_MAP = {'Vol': 'GrayVol', 'CT': 'ThickAvg', 'SA': 'SurfArea'}

def select_features(df, feature_subset):
    if 'ALL' in feature_subset:
        return df.drop('SUBID', axis=1, errors='ignore')
    suffixes = [FEATURE_MAP[f] for f in feature_subset if f in FEATURE_MAP]
    selected_cols = [c for c in df.columns
                     if any(c.endswith(f'_{s}') for s in suffixes) and c != 'SUBID']
    return df[selected_cols]

df_zscore = pd.read_csv(ZSCORE_FILE_PATH)
X_data_df = select_features(df_zscore.copy(), FEATURE_SUBSET)
X_data = X_data_df.values

kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_data)
cluster_centers = kmeans.cluster_centers_

avg_z_scores = cluster_centers.mean(axis=1)
sorted_indices = np.argsort(avg_z_scores)
label_mapping = {sorted_indices[0]: 0, sorted_indices[1]: 1}
standardized_labels = np.array([label_mapping[label] for label in cluster_labels])

subtyping_results = pd.DataFrame({'SUBID': df_zscore['SUBID'].tolist(), 'SUBTYPE_LABEL': standardized_labels})
print(subtyping_results['SUBTYPE_LABEL'].value_counts())

os.makedirs(OUTPUT_DIR, exist_ok=True)
feature_str = '_'.join(FEATURE_SUBSET)
subtyping_results.to_csv(f'{OUTPUT_DIR}/Subtype_aparc_{feature_str}.csv', index=False)
print(f"Saved subtype labels to {OUTPUT_DIR}/Subtype_aparc_{feature_str}.csv")

## Part 3 — Epicenter seeding (GOF)

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from tqdm import tqdm

ZSCORE_PATH = 'output/CABIC_Morpho_Zscore_aparc.csv'
TDC_AVG_MIND_PATH = 'output/TDC_Average_MIND_CABIC_aparc.csv'
SUBJECT_SUBTYPE_PATH = 'data/CABIC_AGE_Sub.xlsx'
OUTPUT_DIR = 'output/Center'
N_REGIONS = 68
N_FEATURES = 1

os.makedirs(OUTPUT_DIR, exist_ok=True)

df_info = pd.read_excel(SUBJECT_SUBTYPE_PATH)
df_info.rename(columns={'SUBID': 'Subject'}, inplace=True)
df_info['Subject'] = df_info['Subject'].astype(str).str.strip()
df_info.set_index('Subject', inplace=True)

df_zscore = pd.read_csv(ZSCORE_PATH)
df_zscore.set_index('SUBID', inplace=True)
df_zscore.index = df_zscore.index.astype(str).str.strip()
target_columns = [c for c in df_zscore.columns if 'GrayVol' in c]
df_zscore = df_zscore[target_columns]

df_avg_mind = pd.read_csv(TDC_AVG_MIND_PATH)
region_names = df_avg_mind.columns.tolist()
mind_matrix = df_avg_mind.values

prediction_patterns = np.zeros((N_REGIONS, N_REGIONS * N_FEATURES))
for i in range(N_REGIONS):
    prediction_patterns[i, :] = np.repeat(mind_matrix[i, :], N_FEATURES)

df_asd_info = df_info[df_info['GROUP'] == 'ASD']
asd_subjects = df_asd_info.index.unique().tolist()
df_asd_zscore = df_zscore[df_zscore.index.isin(asd_subjects)]

gof_matrix = np.zeros((df_asd_zscore.shape[0], N_REGIONS))
asd_sub_ids = df_asd_zscore.index.tolist()
for sub_idx, sub_id in enumerate(tqdm(asd_sub_ids, desc="GOF")):
    actual_z_vector = df_asd_zscore.loc[sub_id].values
    for seed_idx in range(N_REGIONS):
        predicted_pattern = prediction_patterns[seed_idx, :]
        corr, _ = pearsonr(np.nan_to_num(actual_z_vector), np.nan_to_num(predicted_pattern))
        gof_matrix[sub_idx, seed_idx] = corr

df_gof = pd.DataFrame(gof_matrix, index=asd_sub_ids, columns=region_names)
df_gof.index.name = 'SUBID'
df_gof.to_csv(f'{OUTPUT_DIR}/CABIC_ASD_GOF_Scores_aparc.csv')

best_seeds = df_gof.idxmin(axis=1)
seed_counts = best_seeds.value_counts()
total_asd = len(df_gof)

full_region_df = pd.DataFrame(index=region_names)
full_region_df.index.name = 'Brain_Region'
seed_counts_total = full_region_df.join(seed_counts.to_frame(name='Count_Total_ASD'), how='left').fillna(0)
seed_counts_total['Count_Total_ASD'] = seed_counts_total['Count_Total_ASD'].astype(int)
seed_counts_total['Frequency_Total_ASD'] = seed_counts_total['Count_Total_ASD'] / total_asd
seed_counts_total.sort_values('Frequency_Total_ASD', ascending=False, inplace=True)
seed_counts_total.to_csv(f'{OUTPUT_DIR}/CABIC_ASD_Total_Best_Seeds_MinGOF_aparc.csv')

print(f"\nTop 10 epicenters (Min GOF frequency):")
print(seed_counts_total.head(10).to_string())